# Synthetic Rank-One Test

In [31]:
import numpy as np
from ssl_shareability_metric.ssl_encoder_shareability import ssl_encoder_shareability
import torch
from ssl_shareability_metric.shared_encoder import SharedEncoder
from ssl_shareability_metric.seperate_encoder import SeperateEncoder
import copy

torch.manual_seed(42)

## Create rank 1 future representations for synthetic low mid and high sharability

In [32]:
rng = np.random.default_rng(42)
observation_1 = rng.normal(size=(5000, 2))

u_low = np.zeros(2)
v_low = np.zeros(2)

u_low[0] = 1.0
v_low[1] = 1.0

M_low = np.outer(u_low, v_low)
low_observation_2 = observation_1 @ M_low.T

u_mid = np.zeros(2)
v_mid = np.zeros(2)

u_mid[0] = 1.0
v_mid[0] = 0.5
v_mid[1] = np.sqrt(0.75)

M_mid = np.outer(u_mid, v_mid)
mid_observation_2 = observation_1 @ M_mid.T

u_high = np.zeros(2)
v_high = np.zeros(2)

u_high[0] = 1
v_high[0] = 1

M_high = np.outer(u_high, v_high)
high_observation_2 = observation_1 @ M_high.T

print(f"observation 1: {observation_1.shape}")
print(f"low observation 2: {low_observation_2.shape}")
print(f"mid observation 2: {mid_observation_2.shape}")
print(f"high observation 2: {high_observation_2.shape}")

observation 1: (5000, 2)
low observation 2: (5000, 2)
mid observation 2: (5000, 2)
high observation 2: (5000, 2)


## Create splits

In [ ]:
train_split_idx = int(observation_1.shape[0] * 0.70)
val_split_idx = train_split_idx + int(observation_1.shape[0] * 0.10)

train_observation_1 = observation_1[:train_split_idx]
val_observation_1 = observation_1[train_split_idx:val_split_idx]
test_observation_1 = observation_1[val_split_idx:]

train_low_observation_2 = low_observation_2[:train_split_idx]
val_low_observation_2 = low_observation_2[train_split_idx:val_split_idx]
test_low_observation_2 = low_observation_2[val_split_idx:]

train_mid_observation_2 = mid_observation_2[:train_split_idx]
val_mid_observation_2 = mid_observation_2[train_split_idx:val_split_idx]
test_mid_observation_2 = mid_observation_2[val_split_idx:]

train_high_observation_2 = high_observation_2[:train_split_idx]
val_high_observation_2 = high_observation_2[train_split_idx:val_split_idx]
test_high_observation_2 = high_observation_2[val_split_idx:]

print(f"train observation 1: {train_observation_1.shape}")
print(f"val observation 1: {val_observation_1.shape}")
print(f"test observation 1: {test_observation_1.shape}")

print(f"train observation 2 low: {train_low_observation_2.shape}")
print(f"val observation 2 low: {val_low_observation_2.shape}")
print(f"test observation 2 low: {test_low_observation_2.shape}")

print(f"train observation 2 mid: {train_mid_observation_2.shape}")
print(f"val observation 2 mid: {val_mid_observation_2.shape}")
print(f"test futobservation 2ure mid: {test_mid_observation_2.shape}")

print(f"train observation 2 high: {train_high_observation_2.shape}")
print(f"val observation 2 high: {val_high_observation_2.shape}")
print(f"test observation 2 high: {test_high_observation_2.shape}")

train current: (3500, 2)
val current: (500, 2)
test current: (1000, 2)
train future low: (3500, 2)
val future low: (500, 2)
test future low: (1000, 2)
train future mid: (3500, 2)
val future mid: (500, 2)
test future mid: (1000, 2)
train future high: (3500, 2)
val future high: (500, 2)
test future high: (1000, 2)


## Shareability metric

In [34]:
low_shareability = ssl_encoder_shareability(train_observation_1, train_low_observation_2)
mid_shareability = ssl_encoder_shareability(train_observation_1, train_mid_observation_2)
high_shareability = ssl_encoder_shareability(train_observation_1, train_high_observation_2)

print(f"low shareability: {low_shareability}")
print(f"mid shareability: {mid_shareability}")
print(f"high shareability: {high_shareability}")

low shareability: 0.5143347640959772
mid shareability: 0.7449097013893763
high shareability: 0.9997671984073327


## Convert to tensros

In [35]:
train_observation_1_tensor = torch.tensor(train_observation_1, dtype=torch.float32)
val_observation_1_tensor = torch.tensor(val_observation_1, dtype=torch.float32)
test_observation_1_tensor = torch.tensor(test_observation_1, dtype=torch.float32)

train_low_observation_2_tensor = torch.tensor(train_low_observation_2, dtype=torch.float32)
val_low_observation_2_tensor = torch.tensor(val_low_observation_2, dtype=torch.float32)
test_low_observation_2_tensor = torch.tensor(test_low_observation_2, dtype=torch.float32)

train_mid_observation_2_tensor = torch.tensor(train_mid_observation_2, dtype=torch.float32)
val_mid_observation_2_tensor = torch.tensor(val_mid_observation_2, dtype=torch.float32)
test_mid_observation_2_tensor = torch.tensor(test_mid_observation_2, dtype=torch.float32)

train_high_observation_2_tensor = torch.tensor(train_high_observation_2, dtype=torch.float32)
val_high_observation_2_tensor = torch.tensor(val_high_observation_2, dtype=torch.float32)
test_high_observation_2_tensor = torch.tensor(test_high_observation_2, dtype=torch.float32)

## Init encoders and optimizers

In [36]:
low_shared_encoder = SharedEncoder(vector_size=2)
low_shared_optimizer = torch.optim.SGD(low_shared_encoder.parameters(), lr=1e-2)

low_seperate_encoder = SeperateEncoder(vector_size=2)
low_seperate_optimizer = torch.optim.SGD(low_seperate_encoder.parameters(), lr=1e-2)

## Low Rank 1 Case

### Shared

In [37]:
epochs = 100
best_val_loss = float("inf")
best_state = None

for i in range(epochs):
    low_shared_encoder.train()
    Z_x = low_shared_encoder(train_observation_1_tensor)
    Z_y = low_shared_encoder(train_low_observation_2_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    low_shared_optimizer.zero_grad()
    loss.backward()
    low_shared_optimizer.step()
    with torch.no_grad():
        weights = low_shared_encoder.shared.weight
        weights.div_(weights.norm(p=2))
             
    low_shared_encoder.eval()
    with torch.no_grad():
        Z_x = low_shared_encoder(val_observation_1_tensor)
        Z_y = low_shared_encoder(val_low_observation_2_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss = loss.item()
            
    print(f"epoch: {i+1} train: {train_loss} val: {val_loss}")   
            
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(low_shared_encoder.state_dict())

        
if best_state is not None:
    low_shared_encoder.load_state_dict(best_state)

epoch: 1 train: -0.3400357961654663 val: -0.43202823400497437
epoch: 2 train: -0.5341765880584717 val: -0.43204036355018616
epoch: 3 train: -0.5343018174171448 val: -0.432049959897995
epoch: 4 train: -0.534421980381012 val: -0.4320572018623352
epoch: 5 train: -0.534537136554718 val: -0.4320622682571411
epoch: 6 train: -0.5346477627754211 val: -0.4320651888847351
epoch: 7 train: -0.5347537398338318 val: -0.4320662319660187
epoch: 8 train: -0.5348554849624634 val: -0.43206530809402466
epoch: 9 train: -0.5349529981613159 val: -0.43206268548965454
epoch: 10 train: -0.5350465774536133 val: -0.43205851316452026
epoch: 11 train: -0.535136342048645 val: -0.43205273151397705
epoch: 12 train: -0.5352224707603455 val: -0.4320455491542816
epoch: 13 train: -0.5353050827980042 val: -0.43203699588775635
epoch: 14 train: -0.5353842377662659 val: -0.43202731013298035
epoch: 15 train: -0.5354602932929993 val: -0.43201643228530884
epoch: 16 train: -0.5355333089828491 val: -0.4320043623447418
epoch: 17 tr

epoch: 68 train: -0.5370500683784485 val: -0.43096816539764404
epoch: 69 train: -0.5370580554008484 val: -0.4309506118297577
epoch: 70 train: -0.537065863609314 val: -0.4309330880641937
epoch: 71 train: -0.5370731353759766 val: -0.4309157729148865
epoch: 72 train: -0.5370801091194153 val: -0.4308988153934479
epoch: 73 train: -0.5370869040489197 val: -0.43088191747665405
epoch: 74 train: -0.5370933413505554 val: -0.43086540699005127
epoch: 75 train: -0.5370996594429016 val: -0.4308490455150604
epoch: 76 train: -0.5371055603027344 val: -0.43083301186561584
epoch: 77 train: -0.5371113419532776 val: -0.4308170676231384
epoch: 78 train: -0.5371167659759521 val: -0.43080151081085205
epoch: 79 train: -0.5371221303939819 val: -0.4307861626148224
epoch: 80 train: -0.5371272563934326 val: -0.4307708442211151
epoch: 81 train: -0.5371319651603699 val: -0.4307559132575989
epoch: 82 train: -0.5371366739273071 val: -0.43074119091033936
epoch: 83 train: -0.5371410846710205 val: -0.43072667717933655
ep

### Seperate

In [38]:
epochs = 100
best_val_loss = float("inf")
best_state = None

for i in range(epochs):
    low_seperate_encoder.train()
    Z_x, Z_y = low_seperate_encoder(train_observation_1_tensor, train_low_observation_2_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    low_seperate_optimizer.zero_grad()
    loss.backward()
    low_seperate_optimizer.step()
    with torch.no_grad():
        current_weights = low_seperate_encoder.current.weight
        future_weights = low_seperate_encoder.future.weight
        
        current_weights.div_(current_weights.norm(p=2)) 
        future_weights.div_(future_weights.norm(p=2))
             
    low_seperate_encoder.eval()
    with torch.no_grad():
        Z_x, Z_y = low_seperate_encoder(val_observation_1_tensor, val_low_observation_2_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss = loss.item()
            
    print(f"epoch: {i+1} train: {train_loss} val: {val_loss}")   
            
            
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(low_seperate_encoder.state_dict())

        
if best_state is not None:
    low_seperate_encoder.load_state_dict(best_state)

epoch: 1 train: -0.10430647432804108 val: -0.68287593126297
epoch: 2 train: -0.7531251311302185 val: -0.6871053576469421
epoch: 3 train: -0.7579527497291565 val: -0.6912772059440613
epoch: 4 train: -0.7627183794975281 val: -0.6953915953636169
epoch: 5 train: -0.7674221396446228 val: -0.6994490623474121
epoch: 6 train: -0.7720644474029541 val: -0.7034497261047363
epoch: 7 train: -0.7766454815864563 val: -0.707394003868103
epoch: 8 train: -0.7811656594276428 val: -0.7112818956375122
epoch: 9 train: -0.7856249809265137 val: -0.7151143550872803
epoch: 10 train: -0.7900240421295166 val: -0.7188913226127625
epoch: 11 train: -0.794363260269165 val: -0.7226132154464722
epoch: 12 train: -0.7986425161361694 val: -0.7262807488441467
epoch: 13 train: -0.8028627634048462 val: -0.7298939824104309
epoch: 14 train: -0.8070240020751953 val: -0.7334531545639038
epoch: 15 train: -0.8111266493797302 val: -0.7369591593742371
epoch: 16 train: -0.8151711225509644 val: -0.7404119968414307
epoch: 17 train: -0.

### Holdout Set

In [39]:
with torch.no_grad():
    Z_x = low_shared_encoder(test_observation_1_tensor)
    Z_y = low_shared_encoder(test_low_observation_2_tensor)
    low_shared_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"shared loss: {low_shared_loss}")

with torch.no_grad():
    Z_x, Z_y = low_seperate_encoder(test_observation_1_tensor, test_low_observation_2_tensor)
    low_sep_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"seperate loss: {low_sep_loss}")

shared loss: -0.564094603061676
seperate loss: -1.0712391138076782


### Analysis

In [40]:
low_test_result = low_shared_loss / low_sep_loss
print(f"Low test result: {low_test_result}, Low shareability result: {low_shareability}")

Low test result: 0.5265814065933228, Low shareability result: 0.5143347640959772


## Mid Rank One Case

In [41]:
mid_shared_encoder = SharedEncoder(vector_size=2)
mid_shared_optimizer = torch.optim.SGD(mid_shared_encoder.parameters(), lr=1e-2)

mid_seperate_encoder = SeperateEncoder(vector_size=2)
mid_seperate_optimizer = torch.optim.SGD(mid_seperate_encoder.parameters(), lr=1e-2)

In [42]:
epochs = 100
best_val_loss = float("inf")
best_state = None

for i in range(epochs):
    train_loss = 0
    mid_shared_encoder.train()
    Z_x = mid_shared_encoder(train_observation_1_tensor)
    Z_y = mid_shared_encoder(train_mid_observation_2_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss += loss.item()
    mid_shared_optimizer.zero_grad()
    loss.backward()
    mid_shared_optimizer.step()
    with torch.no_grad():
        weights = mid_shared_encoder.shared.weight
        weights.div_(weights.norm(p=2))
             
    val_loss = 0
    mid_shared_encoder.eval()
    with torch.no_grad():
        Z_x = mid_shared_encoder(val_observation_1_tensor)
        Z_y = mid_shared_encoder(val_mid_observation_2_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss += loss.item()
            
    print(f"epoch: {i+1} train: {train_loss} val: {val_loss}")   
            
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(mid_shared_encoder.state_dict())
    
if best_state is not None:
    mid_shared_encoder.load_state_dict(best_state)
        

epoch: 1 train: -0.07020483911037445 val: -0.20191021263599396
epoch: 2 train: -0.24240289628505707 val: -0.20281778275966644
epoch: 3 train: -0.24347831308841705 val: -0.20368964970111847
epoch: 4 train: -0.24451126158237457 val: -0.20452706515789032
epoch: 5 train: -0.2455032616853714 val: -0.20533131062984467
epoch: 6 train: -0.24645580351352692 val: -0.20610378682613373
epoch: 7 train: -0.24737054109573364 val: -0.20684559643268585
epoch: 8 train: -0.24824878573417664 val: -0.20755790174007416
epoch: 9 train: -0.24909202754497528 val: -0.20824190974235535
epoch: 10 train: -0.24990156292915344 val: -0.20889857411384583
epoch: 11 train: -0.2506786584854126 val: -0.2095290869474411
epoch: 12 train: -0.2514245808124542 val: -0.2101343721151352
epoch: 13 train: -0.252140611410141 val: -0.2107154130935669
epoch: 14 train: -0.2528277337551117 val: -0.211273193359375
epoch: 15 train: -0.2534872591495514 val: -0.21180853247642517


epoch: 16 train: -0.25412020087242126 val: -0.2123224288225174
epoch: 17 train: -0.2547276020050049 val: -0.21281567215919495
epoch: 18 train: -0.25531038641929626 val: -0.2132890820503235
epoch: 19 train: -0.25586968660354614 val: -0.21374335885047913
epoch: 20 train: -0.256406307220459 val: -0.2141793668270111
epoch: 21 train: -0.2569211721420288 val: -0.21459776163101196
epoch: 22 train: -0.25741517543792725 val: -0.2149992436170578
epoch: 23 train: -0.257889062166214 val: -0.21538449823856354
epoch: 24 train: -0.2583436667919159 val: -0.2157541662454605
epoch: 25 train: -0.2587798237800598 val: -0.21610885858535767
epoch: 26 train: -0.2591981887817383 val: -0.21644920110702515
epoch: 27 train: -0.2595995366573334 val: -0.21677576005458832
epoch: 28 train: -0.2599845230579376 val: -0.21708904206752777
epoch: 29 train: -0.2603537440299988 val: -0.2173895686864853
epoch: 30 train: -0.260707825422287 val: -0.2176779955625534
epoch: 31 train: -0.2610475420951843 val: -0.2179545909166336

In [43]:
epochs = 100
best_val_loss = float("inf")
best_state = None

for i in range(epochs):
    mid_seperate_encoder.train()
    Z_x, Z_y = mid_seperate_encoder(train_observation_1_tensor, train_mid_observation_2_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    mid_seperate_optimizer.zero_grad()
    loss.backward()
    mid_seperate_optimizer.step()
    with torch.no_grad():
        current_weights = mid_seperate_encoder.current.weight
        future_weights = mid_seperate_encoder.future.weight
        
        current_weights.div_(current_weights.norm(p=2)) 
        future_weights.div_(future_weights.norm(p=2))
             
    mid_seperate_encoder.eval()
    with torch.no_grad():
        Z_x, Z_y = mid_seperate_encoder(val_observation_1_tensor, val_mid_observation_2_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss = loss.item()
            
    print(f"epoch: {i+1} train: {train_loss} val: {val_loss}")   
            
    if (val_loss) < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(mid_seperate_encoder.state_dict())
    
if best_state is not None:
    mid_seperate_encoder.load_state_dict(best_state)

epoch: 1 train: -0.09521632641553879 val: -0.16120241582393646
epoch: 2 train: -0.19472184777259827 val: -0.16978274285793304
epoch: 3 train: -0.20495688915252686 val: -0.17833097279071808
epoch: 4 train: -0.21515342593193054 val: -0.18684552609920502
epoch: 5 train: -0.22530953586101532 val: -0.19532500207424164
epoch: 6 train: -0.23542356491088867 val: -0.20376797020435333


epoch: 7 train: -0.2454938441514969 val: -0.21217304468154907
epoch: 8 train: -0.25551870465278625 val: -0.22053895890712738
epoch: 9 train: -0.26549652218818665 val: -0.2288641631603241
epoch: 10 train: -0.2754257023334503 val: -0.2371474653482437
epoch: 11 train: -0.28530460596084595 val: -0.24538755416870117
epoch: 12 train: -0.2951318025588989 val: -0.25358322262763977
epoch: 13 train: -0.30490565299987793 val: -0.26173317432403564
epoch: 14 train: -0.3146248459815979 val: -0.2698362171649933
epoch: 15 train: -0.3242878019809723 val: -0.2778911888599396
epoch: 16 train: -0.3338932394981384 val: -0.285896897315979
epoch: 17 train: -0.3434397280216217 val: -0.2938523292541504
epoch: 18 train: -0.35292598605155945 val: -0.3017563223838806
epoch: 19 train: -0.362350732088089 val: -0.3096078634262085
epoch: 20 train: -0.37171271443367004 val: -0.3174058794975281
epoch: 21 train: -0.38101059198379517 val: -0.32514944672584534
epoch: 22 train: -0.39024338126182556 val: -0.3328376710414886

In [44]:
with torch.no_grad():
    Z_x = mid_shared_encoder(test_observation_1_tensor)
    Z_y = mid_shared_encoder(test_mid_observation_2_tensor)
    mid_shared_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"shared loss: {mid_shared_loss}")

with torch.no_grad():
    Z_x, Z_y = mid_seperate_encoder(test_observation_1_tensor, test_mid_observation_2_tensor)
    mid_sep_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"seperate loss: {mid_sep_loss}")

shared loss: -0.2984672486782074
seperate loss: -0.9345722794532776


In [45]:
mid_test_result = mid_shared_loss / mid_sep_loss
print(f"Mid test result: {mid_test_result}, Mid shareability result: {mid_shareability}")

Mid test result: 0.3193624019622803, Mid shareability result: 0.7449097013893763


In [52]:
w = mid_shared_encoder.shared.weight.detach().cpu().numpy().flatten()

q_plus = u_mid + v_mid
q_plus = q_plus / np.linalg.norm(q_plus)

q_minus = u_mid - v_mid
q_minus = q_minus / np.linalg.norm(q_minus)

print("alignment with +:", abs(np.dot(w, q_plus)))
print("alignment with -:", abs(np.dot(w, q_minus)))

alignment with +: 0.025563661599698195
alignment with -: 0.9996732354091447


## High Rank One Case

In [46]:
high_shared_encoder = SharedEncoder(vector_size=2)
high_shared_optimizer = torch.optim.SGD(high_shared_encoder.parameters(), lr=1e-2)

high_seperate_encoder = SeperateEncoder(vector_size=2)
high_seperate_optimizer = torch.optim.SGD(high_seperate_encoder.parameters(), lr=1e-2)

In [47]:
epochs = 100
best_val_loss = float("inf")
best_state = None

for i in range(epochs):
    high_shared_encoder.train()
    Z_x = high_shared_encoder(train_observation_1_tensor)
    Z_y = high_shared_encoder(train_high_observation_2_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    high_shared_optimizer.zero_grad()
    loss.backward()
    high_shared_optimizer.step()
    with torch.no_grad():
        weights = high_shared_encoder.shared.weight
        weights.div_(weights.norm(p=2))
             
    high_shared_encoder.eval()
    with torch.no_grad():
        Z_x = high_shared_encoder(val_observation_1_tensor)
        Z_y = high_shared_encoder(val_high_observation_2_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss = loss.item()
            
    print(f"epoch: {i+1} train: {train_loss} val: {val_loss}")   
            
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(high_shared_encoder.state_dict())

        
    
if best_state is not None:
    high_shared_encoder.load_state_dict(best_state)

epoch: 1 train: -0.269234299659729 val: -0.9349837899208069
epoch: 2 train: -0.9554225206375122 val: -0.936204195022583
epoch: 3 train: -0.9563818573951721 val: -0.9373858571052551
epoch: 4 train: -0.9573065042495728 val: -0.9385298490524292
epoch: 5 train: -0.958197832107544 val: -0.939637303352356
epoch: 6 train: -0.9590563774108887 val: -0.9407095909118652
epoch: 7 train: -0.9598836898803711 val: -0.9417476058006287
epoch: 8 train: -0.9606809020042419 val: -0.9427524209022522
epoch: 9 train: -0.9614487290382385 val: -0.9437254667282104
epoch: 10 train: -0.9621888995170593 val: -0.9446674585342407
epoch: 11 train: -0.962901771068573 val: -0.9455795288085938
epoch: 12 train: -0.9635884761810303 val: -0.9464622735977173
epoch: 13 train: -0.9642496109008789 val: -0.9473171234130859
epoch: 14 train: -0.964886486530304 val: -0.9481448531150818
epoch: 15 train: -0.9655002355575562 val: -0.9489461779594421
epoch: 16 train: -0.9660910367965698 val: -0.9497222900390625
epoch: 17 train: -0.966

In [48]:
epochs = 100
best_val_loss = float("inf")
best_state = None

for i in range(epochs):
    high_seperate_encoder.train()
    Z_x, Z_y = high_seperate_encoder(train_observation_1_tensor, train_high_observation_2_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    high_seperate_optimizer.zero_grad()
    loss.backward()
    high_seperate_optimizer.step()
    with torch.no_grad():
        current_weights = high_seperate_encoder.current.weight
        future_weights = high_seperate_encoder.future.weight
        
        current_weights.div_(current_weights.norm(p=2)) 
        future_weights.div_(future_weights.norm(p=2))
             
    val_loss = 0
    high_seperate_encoder.eval()
    with torch.no_grad():
        Z_x, Z_y = high_seperate_encoder(val_observation_1_tensor, val_high_observation_2_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss = loss.item()
            
    print(f"epoch: {i+1} train: {train_loss} val: {val_loss}")   
            
            
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(high_seperate_encoder.state_dict())

    
if best_state is not None:
    high_seperate_encoder.load_state_dict(best_state)

epoch: 1 train: -0.18069656193256378 val: -0.938622236251831
epoch: 2 train: -0.9179326295852661 val: -0.939531147480011
epoch: 3 train: -0.9190921187400818 val: -0.9404218792915344
epoch: 4 train: -0.9202315807342529 val: -0.9412949681282043
epoch: 5 train: -0.9213514924049377 val: -0.9421505331993103
epoch: 6 train: -0.9224521517753601 val: -0.9429885745048523
epoch: 7 train: -0.9235336780548096 val: -0.9438100457191467
epoch: 8 train: -0.9245965480804443 val: -0.9446146488189697
epoch: 9 train: -0.925640881061554 val: -0.945402979850769
epoch: 10 train: -0.9266670346260071 val: -0.9461749196052551
epoch: 11 train: -0.9276753067970276 val: -0.9469314217567444
epoch: 12 train: -0.9286659359931946 val: -0.9476721286773682
epoch: 13 train: -0.9296392202377319 val: -0.9483976364135742
epoch: 14 train: -0.9305955171585083 val: -0.9491078853607178
epoch: 15 train: -0.9315348267555237 val: -0.9498037099838257
epoch: 16 train: -0.9324578642845154 val: -0.9504847526550293
epoch: 17 train: -0.

In [49]:
with torch.no_grad():
    Z_x = high_shared_encoder(test_observation_1_tensor)
    Z_y = high_shared_encoder(test_high_observation_2_tensor)
    high_shared_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"shared loss: {high_shared_loss}")

with torch.no_grad():
    Z_x, Z_y = high_seperate_encoder(test_observation_1_tensor, test_high_observation_2_tensor)
    high_sep_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"seperate loss: {high_sep_loss}")

shared loss: -0.9703022241592407
seperate loss: -0.9635375142097473


In [50]:
high_test_result = high_shared_loss / high_sep_loss
print(f"High test result: {high_test_result}, High shareability result: {high_shareability}")

High test result: 1.0070207118988037, High shareability result: 0.9997671984073327


In [51]:
print(f"Low test result: {low_test_result}, Low shareability result: {low_shareability}")
print(f"Mid test result: {mid_test_result}, Mid shareability result: {mid_shareability}")
print(f"High test result: {high_test_result}, High shareability result: {high_shareability}")

print(f"Low Error: {(abs(low_test_result - low_shareability) / low_shareability * 100):.2f}%")
print(f"Mid Error: {(abs(mid_test_result - mid_shareability) / mid_shareability * 100):.2f}%")
print(f"High Error: {(abs(high_test_result - high_shareability) / high_shareability * 100):.2f}%")

Low test result: 0.5265814065933228, Low shareability result: 0.5143347640959772
Mid test result: 0.3193624019622803, Mid shareability result: 0.7449097013893763
High test result: 1.0070207118988037, High shareability result: 0.9997671984073327
Low Error: 2.38%
Mid Error: 57.13%
High Error: 0.73%


restarts + document the mathmatical reason clearly